In [1]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline

from google.colab import drive
import warnings

warnings.filterwarnings('ignore')
print('Libraries imported successfully')

Libraries imported successfully


In [2]:
# Google drive connection

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Loading preprocessor, labels, variable names and losgitic regresion model (balanced)

preprocessor = joblib.load(
    "/content/drive/MyDrive/Credit_Risk_Project/models/preprocessor.pkl"
)

feature_names = pd.read_csv(
    "/content/drive/MyDrive/Credit_Risk_Project/data/processed/feature_names.csv"
)

logistic_balanced_model = joblib.load(
    "/content/drive/MyDrive/Credit_Risk_Project/models/logistic_balanced_model.pkl"
)

In [4]:
# Building pipeline
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", logistic_balanced_model)
])

In [5]:
# Saving pipeline
joblib.dump(
    final_pipeline,
    "/content/drive/MyDrive/Credit_Risk_Project/models/credit_default_pipeline.pkl"
)

['/content/drive/MyDrive/Credit_Risk_Project/models/credit_default_pipeline.pkl']

In [6]:
# Loading pipeline
pipeline = joblib.load(
    "/content/drive/MyDrive/Credit_Risk_Project/models/credit_default_pipeline.pkl"
)

In [7]:
def predict_default_risk(client_data, pipeline, threshold=0.50):

    client_df = pd.DataFrame([client_data])
    probability = pipeline.predict_proba(client_df)[0,1]
    prediction = int(probability >= threshold)

    if probability < 0.30:
        risk = "Low"

    elif probability < 0.60:
        risk = "Medium"

    else:
        risk = "High"

    print("="*50)
    print("CREDIT RISK ASSESSMENT")
    print("="*50)

    print(f"\nProbability of Default : {probability:.2%}")
    print(f"Predicted Class        : {prediction}")
    print(f"Risk Level             : {risk}")

    return probability, prediction, risk

In [8]:
def identify_risk_factors(client):

    factors = []

    # Age
    if client["Age"] < 30:
        factors.append("- Young age")

    # Income
    if client["Income"] < 50000:
        factors.append("- Low income")

    # Loan amount
    if client["LoanAmount"] > 150000:
        factors.append("- High loan amount")

    # Credit score
    if client["CreditScore"] < 600:
        factors.append("- Low credit score")

    # Months employed
    if client["MonthsEmployed"] < 24:
        factors.append("- Short employment history")

    # Number of credit lines
    if client["NumCreditLines"] >= 4:
        factors.append("- Many credit lines")

    # Interest rate
    if client["InterestRate"] >= 15:
        factors.append("- High interest rate")

    # DTI
    if client["DTIRatio"] >= 0.50:
        factors.append("- High debt-to-income ratio")

    # Employment type
    if client["EmploymentType"] == "Unemployed":
        factors.append("- Unemployed")

    # Co-signer
    if client["HasCoSigner"] == "No":
        factors.append("- No co-signer")

    return factors

In [9]:
def generate_recommendation(probability):

    if probability >= 0.70:

        return (
            "The applicant presents a HIGH credit risk. "
            "A detailed credit review is recommended before loan approval."
        )

    elif probability >= 0.40:

        return (
            "The applicant presents a MODERATE credit risk. "
            "Additional documentation or guarantees are recommended."
        )

    else:

        return (
            "The applicant presents a LOW credit risk according to the model."
        )

In [10]:
EXPECTED_COLUMNS = [
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "LoanTerm",
    "DTIRatio",
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "HasMortgage",
    "HasDependents",
    "LoanPurpose",
    "HasCoSigner"
]

In [11]:
VALID_CATEGORIES = {

    "Education": [
        "Bachelor's",
        "High School",
        "Master's",
        "PhD"
    ],

    "EmploymentType": [
        "Full-time",
        "Part-time",
        "Self-employed",
        "Unemployed"
    ],

    "MaritalStatus": [
        "Single",
        "Married",
        "Divorced"
    ],

    "LoanPurpose": [
        "Auto",
        "Business",
        "Education",
        "Home",
        "Other"
    ]
}

In [12]:
def validate_client(client):

    # -----------------------------
    # Verificar columnas faltantes
    # -----------------------------

    missing = [
        col for col in EXPECTED_COLUMNS
        if col not in client
    ]

    if missing:

        raise ValueError(
            f"Missing variables: {missing}"
        )

    # -----------------------------
    # Variables binarias
    # -----------------------------

    binary_mapping = {

        "Yes":1,
        "No":0,

        "yes":1,
        "no":0,

        "YES":1,
        "NO":0,

        True:1,
        False:0,

        1:1,
        0:0
    }

    for feature in [

        "HasMortgage",
        "HasDependents",
        "HasCoSigner"

    ]:

        value = client[feature]

        if value not in binary_mapping:

            raise ValueError(

                f"{feature} must be Yes/No or 1/0."

            )

        client[feature] = binary_mapping[value]

    # -----------------------------
    # Variables categóricas
    # -----------------------------

    for feature, valid_values in VALID_CATEGORIES.items():

        if client[feature] not in valid_values:

            raise ValueError(

                f"{feature} must be one of: {valid_values}"

            )

    # -----------------------------
    # Variables numéricas
    # -----------------------------

    numeric_variables = [

        "Age",
        "Income",
        "LoanAmount",
        "CreditScore",
        "MonthsEmployed",
        "NumCreditLines",
        "InterestRate",
        "LoanTerm",
        "DTIRatio"

    ]

    for feature in numeric_variables:

        try:

            client[feature] = float(client[feature])

        except:

            raise ValueError(

                f"{feature} must be numeric."

            )

    # -----------------------------
    # Business Rules Validation
    # -----------------------------

    if not (18 <= client["Age"] <= 100):
      raise ValueError(
        "Age must be between 18 and 100 years."
    )

    if client["Income"] <= 0:
      raise ValueError(
        "Income must be greater than 0."
    )

    if client["LoanAmount"] <= 0:
      raise ValueError(
        "LoanAmount must be greater than 0."
    )

    if not (300 <= client["CreditScore"] <= 850):
      raise ValueError(
        "CreditScore must be between 300 and 850."
    )

    if client["MonthsEmployed"] < 0:
      raise ValueError(
        "MonthsEmployed cannot be negative."
    )

    if client["NumCreditLines"] < 0:
      raise ValueError(
        "NumCreditLines cannot be negative."
    )

    if not (0 <= client["InterestRate"] <= 100):
      raise ValueError(
        "InterestRate must be between 0 and 100."
    )

    if client["LoanTerm"] <= 0:
      raise ValueError(
        "LoanTerm must be greater than 0."
    )

    if not (0 <= client["DTIRatio"] <= 1):
      raise ValueError(
        "DTIRatio must be between 0 and 1."
    )

    return client

In [13]:
def generate_credit_report(client, pipeline, threshold=0.50):

    client = client.copy()

    try:
      client = validate_client(client)

    except ValueError as e:

      print("="*60)
      print("INPUT VALIDATION ERROR")
      print("="*60)

      print(f"\n{e}")

      return

    client_df = pd.DataFrame([client])

    probability = pipeline.predict_proba(client_df)[0,1]

    prediction = int(probability >= threshold)

    if probability < 0.30:
        risk = "LOW"

    elif probability < 0.60:
        risk = "MEDIUM"

    else:
        risk = "HIGH"

    print("="*60)
    print("             CREDIT DEFAULT RISK REPORT")
    print("="*60)

    print(f"\nProbability of Default : {probability:.2%}")
    print(f"Predicted Class        : {prediction}")
    print(f"Risk Level             : {risk}")
    print("\nMain Risk Factors")
    print("-"*60)

    factors = identify_risk_factors(client)

    if len(factors) == 0:
        print("✓ No major risk factors detected.")

    else:

        for factor in factors:
            print(factor)

    print("\nRecommendation")
    print("-"*60)
    print(generate_recommendation(probability))
    print("\n" + "="*60)

In [14]:
# Create new client/borrower
client = {

    "Age": 23,
    "Income":40000,
    "LoanAmount":15000,
    "CreditScore":400,
    "MonthsEmployed":24,
    "NumCreditLines":4,
    "InterestRate":14,
    "LoanTerm":48,
    "DTIRatio":0.48,

    "Education":"Bachelor's",
    "EmploymentType":"Unemployed",
    "MaritalStatus":"Single",

    "HasMortgage":"No",
    "HasDependents":"No",
    "LoanPurpose":"Business",
    "HasCoSigner":"No"
}

In [15]:
generate_credit_report(
    client,
    pipeline
)

             CREDIT DEFAULT RISK REPORT

Probability of Default : 84.68%
Predicted Class        : 1
Risk Level             : HIGH

Main Risk Factors
------------------------------------------------------------
- Young age
- Low income
- Low credit score
- Many credit lines
- Unemployed

Recommendation
------------------------------------------------------------
The applicant presents a HIGH credit risk. A detailed credit review is recommended before loan approval.

